# Week 2 Local Test Session\n

- Test scope: Week 2 API async flow (local fallback) + Admin API smoke
- Session date: 2026-03-11
- Phase: Tuan 2 (T2.2, T2.3 local-first)
- Reason: Cloud Run build chưa cần, tiep tuc test local truoc

In [7]:
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
import sys

from fastapi.testclient import TestClient

# Ensure repository root is importable when notebook runs from notebooks/tests.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.api.routes.generation as generation_route
import src.api.routes.admin as admin_route
from src.main import app

jobs = {}
docs = {}


async def fake_create_job(request_id, user_id, request_data):
    now = datetime.now(timezone.utc)
    jobs[request_id] = {
        "request_id": request_id,
        "user_id": user_id,
        "status": "processing",
        "request": request_data,
        "created_at": now,
        "updated_at": now,
    }
    return jobs[request_id]


async def fake_get_job(request_id):
    return jobs.get(request_id)


async def fake_execute_generation_job(request_id):
    job = jobs[request_id]
    now = datetime.now(timezone.utc)
    job["status"] = "completed"
    job["completed_at"] = now
    job["updated_at"] = now
    job["content"] = {
        "request_id": request_id,
        "user_id": job["user_id"],
        "generated_at": now.isoformat(),
        "content": {"quiz": [{"id": "q1", "question": "2+2=?"}]},
        "metadata": {
            "total_generated": 1,
            "total_passed_review": 1,
            "total_rejected": 0,
            "generation_time_seconds": 0.01,
            "model_used": "mock-local",
            "vertex_search_queries": 1,
            "game_types_generated": ["quiz"],
            "cost_estimate_usd": None,
        },
    }
    return {"ok": True, "request_id": request_id, "status": "completed"}


def fake_settings():
    return SimpleNamespace(
        cloud_run_base_url=None,
        cloud_tasks_invoker_service_account=None,
        local_async_mode=True,
    )


generation_route.get_settings = fake_settings
generation_route.create_job = fake_create_job
generation_route.get_job = fake_get_job
generation_route.execute_generation_job = fake_execute_generation_job


def fake_upload_document(**kwargs):
    doc_id = kwargs.get("session_id", "session")
    return {
        "gcs_uri": f"gs://documents-development-bucket/system/2026-03-11/{doc_id}/{kwargs.get('filename', 'f.pdf')}",
        "gcs_path": f"system/2026-03-11/{doc_id}/{kwargs.get('filename', 'f.pdf')}",
        "metadata_user_id": "__system__",
        "upload_date": "2026-03-11",
        "size_bytes": len(kwargs.get("file_content", b"")),
    }


def fake_trigger_import(**kwargs):
    return "operations/mock-import"


async def fake_save_document_record(
    doc_id, user_id, gcs_uri, filename, scope="user", **extra
):
    docs[doc_id] = {
        "doc_id": doc_id,
        "user_id": user_id,
        "gcs_uri": gcs_uri,
        "filename": filename,
        "scope": scope,
        **extra,
    }
    return docs[doc_id]


async def fake_list_document_records(scope=None, user_id=None, limit=100):
    items = list(docs.values())
    if scope:
        items = [d for d in items if d.get("scope") == scope]
    if user_id:
        items = [d for d in items if d.get("user_id") == user_id]
    return items[:limit]


async def fake_get_document_record(doc_id):
    return docs.get(doc_id)


async def fake_delete_document_record(doc_id):
    docs.pop(doc_id, None)
    return True


def fake_delete_document(gcs_path):
    return True


admin_route.upload_document = fake_upload_document
admin_route.trigger_import = fake_trigger_import
admin_route.save_document_record = fake_save_document_record
admin_route.list_document_records = fake_list_document_records
admin_route.get_document_record = fake_get_document_record
admin_route.delete_document_record = fake_delete_document_record
admin_route.delete_document = fake_delete_document

client = TestClient(app)
print("SETUP_OK", repo_root)

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SETUP_OK /home/sakana/Code/ManifoldsTeam/AIServices


In [8]:
payload = {
    "user_id": "u-local-001",
    "topic": "Dao ham",
    "game_types": ["quiz"],
    "num_questions": 3,
    "difficulty": "medium",
    "doc_scope": "all",
    "language": "vi",
}

r = client.post("/api/v1/generate", json=payload)
assert r.status_code == 202, r.text
request_id = r.json()["request_id"]

r2 = client.get(f"/api/v1/generations/{request_id}")
assert r2.status_code == 200, r2.text
assert r2.json()["status"] == "completed", r2.json()
print("T2.3_LOCAL_ASYNC_OK", request_id)

Using selector: EpollSelector
Using selector: EpollSelector
T2.3_LOCAL_ASYNC_OK 99e3be72-0905-40a9-be60-7e106eacb00c


In [9]:
files = {"file": ("sys-test.pdf", b"%PDF-1.4 test file", "application/pdf")}
data = {"subject": "toan", "grade": "lop-10"}

up = client.post("/api/v1/admin/documents/upload", files=files, data=data)
assert up.status_code == 201, up.text
doc_id = up.json()["document_id"]
assert up.json()["user_id"] == "__system__"

ls = client.get("/api/v1/admin/documents")
assert ls.status_code == 200, ls.text
assert ls.json()["total"] >= 1

dl = client.delete(f"/api/v1/admin/documents/{doc_id}")
assert dl.status_code == 204, dl.text
print("T2.2_ADMIN_API_OK", doc_id)

Using selector: EpollSelector
Calling on_part_begin with no data
Calling on_header_field with data[36:55]
Calling on_header_value with data[57:82]
Calling on_header_end with no data
Calling on_headers_finished with no data
Calling on_part_data with data[86:90]
Calling on_part_end with no data
Calling on_part_begin with no data
Calling on_header_field with data[128:147]
Calling on_header_value with data[149:172]
Calling on_header_end with no data
Calling on_headers_finished with no data
Calling on_part_data with data[176:182]
Calling on_part_end with no data
Calling on_part_begin with no data
Calling on_header_field with data[220:239]
Calling on_header_value with data[241:288]
Calling on_header_end with no data
Calling on_header_field with data[290:302]
Calling on_header_value with data[304:319]
Calling on_header_end with no data
Calling on_headers_finished with no data
Calling on_part_data with data[323:341]
Calling on_part_end with no data
Calling on_end with no data
Using selector: E

In [10]:
summary = {
    "t23_local_async": len(jobs) >= 1 and all(j.get("status") == "completed" for j in jobs.values()),
    "t22_admin_cleanup": len(docs) == 0,
}
print("SUMMARY", summary)
assert summary["t23_local_async"]
assert summary["t22_admin_cleanup"]
print("WEEK2_LOCAL_TEST_SESSION_PASS")

SUMMARY {'t23_local_async': True, 't22_admin_cleanup': True}
WEEK2_LOCAL_TEST_SESSION_PASS


In [11]:
import src.graph.nodes.formatter as formatter_module
from src.api.schemas.game_content import DifficultyLevel, GameType
from src.api.schemas.requests import DocScope, GenerationRequest


class DummyLLM:
    pass


async def fake_quiz(items, llm):
    return [{"id": "quiz-1", "question": "Q", "options": [], "correct_answer_index": 0, "explanation": "E", "difficulty": "medium", "topic": "T"}]


async def fake_flash(items, llm):
    return [{"id": "flash-1", "front": "F", "back": "B", "topic": "T", "difficulty": "medium", "tags": []}]


async def fake_blank(items, llm):
    return [{"id": "blank-1", "template": "A ___ B", "blanks": [{"position": 0, "correct_answer": "x", "hint": None}], "explanation": "E", "difficulty": "medium", "topic": "T"}]


formatter_module._get_llm = lambda: DummyLLM()
formatter_module._format_quizzes = fake_quiz
formatter_module._format_flashcards = fake_flash
formatter_module._format_fill_blanks = fake_blank

req = GenerationRequest(
    user_id="u-local-001",
    topic="Dao ham",
    game_types=[GameType.QUIZ, GameType.FLASHCARD, GameType.FILL_BLANK],
    num_questions=1,
    difficulty=DifficultyLevel.MEDIUM,
    doc_scope=DocScope.ALL,
    language="vi",
)
state = {
    "request": req,
    "reviewed_items": [
        {
            "question": "2+2=?",
            "answer": "4",
            "explanation": "co ban",
            "topic": "So hoc",
            "difficulty": "medium",
            "computation_trace": None,
        }
    ],
    "rejected_items": [],
}

out = await formatter_module.formatter_node(state)
content = out["final_output"].content
assert set(content.keys()) == {"quiz", "flashcard", "fill_blank"}
assert len(content["quiz"]) == 1
assert len(content["flashcard"]) == 1
assert len(content["fill_blank"]) == 1
print("T2.4_FORMATTER_GAME_TYPES_OK")

2026-03-11T08:38:40.059765Z [info     ] formatter_starting             game_types=['quiz', 'flashcard', 'fill_blank'] items_count=1 user_id=u-local-001
2026-03-11T08:38:40.060817Z [info     ] formatter_completed            game_types={'quiz': 1, 'flashcard': 1, 'fill_blank': 1} generation_time=0.0010380744934082031
T2.4_FORMATTER_GAME_TYPES_OK
